# 02 - MS-ASL Pilot Download and Clip Extraction

Notebook pobiera i wycina klipy MS-ASL z URL.

Zakres pilota:
- 100 rekordow lacznie,
- rekordy 1-50: wycinanie po klatkach,
- rekordy 51-100: wycinanie po czasie (`start_time`/`end_time`),
- zapis klipow do `videos_MS_ASL/`,
- zapis statusow do CSV manifestu.

In [18]:
from __future__ import annotations

from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
import hashlib
import json
import shutil
import subprocess
import tempfile
from typing import Any

import cv2
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)

In [ ]:
PROJECT_ROOT = Path.cwd().resolve().parent.parent if Path.cwd().name == 'MS-ASL_EDA' else (Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve())
MSASL_DIR = PROJECT_ROOT / 'MS-ASL'
TRAIN_JSON = MSASL_DIR / 'MSASL_train.json'
VAL_JSON = MSASL_DIR / 'MSASL_val.json'
TEST_JSON = MSASL_DIR / 'MSASL_test.json'

OUTPUT_VIDEO_DIR = MSASL_DIR / 'videos_MS_ASL_pilot'
OUTPUT_MANIFEST_CSV = MSASL_DIR / 'msasl_download_manifest_compare10.csv'

# Porownanie A/B: te same 10 rekordow w 2 trybach (frames i seconds).
COMPARE_ROWS = 10
MAX_WORKERS = 4
OVERWRITE = False

YTDLP_QUALITY = 'best[height<=720]/best[height<=480]/best'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('MSASL_DIR exists:', MSASL_DIR.exists())
print('OUTPUT_VIDEO_DIR:', OUTPUT_VIDEO_DIR)
print('OUTPUT_MANIFEST_CSV:', OUTPUT_MANIFEST_CSV)
print('COMPARE_ROWS:', COMPARE_ROWS, '| MAX_WORKERS:', MAX_WORKERS)

PROJECT_ROOT: D:\college\sem_mag_1\szum
MSASL_DIR exists: True
OUTPUT_VIDEO_DIR: D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL
OUTPUT_MANIFEST_CSV: D:\college\sem_mag_1\szum\MS-ASL\msasl_download_manifest_compare10.csv
COMPARE_ROWS: 10 | MAX_WORKERS: 4


In [20]:
def check_required_tools() -> None:
    missing = []
    if shutil.which('yt-dlp') is None:
        missing.append('yt-dlp')
    if shutil.which('ffmpeg') is None:
        missing.append('ffmpeg')

    if missing:
        raise EnvironmentError(f'Missing required tools in PATH: {missing}')

    print('Tools OK: yt-dlp, ffmpeg')


def load_split(path: Path, split_name: str) -> pd.DataFrame:
    with path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    df['split'] = split_name
    return df


def normalize_url(raw_url: Any) -> str:
    url = str(raw_url).strip()
    if not url:
        return ''
    low = url.lower()
    if low.startswith('http://') or low.startswith('https://'):
        return url
    if low.startswith('www.'):
        return 'https://' + url
    if 'youtube.com/' in low or 'youtu.be/' in low:
        return 'https://' + url
    return url


def make_pair_id(row: pd.Series) -> str:
    # pair_id jest wspolny dla frames i seconds (bez trim_mode).
    key = '|'.join([
        str(row.get('url_norm', '')),
        str(row.get('clean_text', '')).strip().lower(),
        str(row.get('split', '')),
        str(row.get('start', '')),
        str(row.get('end', '')),
        str(row.get('start_time', '')),
        str(row.get('end_time', '')),
    ])
    return hashlib.sha1(key.encode('utf-8')).hexdigest()[:16]


def make_clip_id(row: pd.Series) -> str:
    # clip_id rozroznia warianty trims dla tego samego pair_id.
    key = '|'.join([
        str(row.get('pair_id', '')),
        str(row.get('trim_mode', '')),
    ])
    return hashlib.sha1(key.encode('utf-8')).hexdigest()[:16]


def yt_dlp_download(url: str, temp_dir: Path) -> Path:
    output_template = str(temp_dir / 'raw.%(ext)s')
    cmd = [
        'yt-dlp',
        '--no-playlist',
        '-f', YTDLP_QUALITY,
        '-o', output_template,
        url,
    ]
    subprocess.run(cmd, check=True, capture_output=True, text=True)

    candidates = sorted(temp_dir.glob('raw.*'))
    if not candidates:
        raise FileNotFoundError('yt-dlp finished but no output file found')
    return candidates[0]


def trim_by_seconds(input_path: Path, output_path: Path, start_s: float, end_s: float) -> None:
    if end_s <= start_s:
        raise ValueError('Invalid second range: end_time <= start_time')

    cmd = [
        'ffmpeg', '-y',
        '-ss', f'{start_s:.6f}',
        '-to', f'{end_s:.6f}',
        '-i', str(input_path),
        '-an',
        '-c:v', 'libx264',
        '-preset', 'fast',
        '-crf', '23',
        '-movflags', '+faststart',
        str(output_path),
    ]
    subprocess.run(cmd, check=True, capture_output=True, text=True)


def trim_by_frames(input_path: Path, output_path: Path, start_f: int, end_f: int) -> None:
    if end_f < start_f:
        raise ValueError('Invalid frame range: end < start')

    vf = f"select='between(n\,{start_f}\,{end_f})',setpts=N/FRAME_RATE/TB"
    cmd = [
        'ffmpeg', '-y',
        '-i', str(input_path),
        '-vf', vf,
        '-an',
        '-c:v', 'libx264',
        '-preset', 'fast',
        '-crf', '23',
        '-movflags', '+faststart',
        str(output_path),
    ]
    subprocess.run(cmd, check=True, capture_output=True, text=True)


def probe_video(video_path: Path) -> tuple[float | None, int | None, int | None, int | None]:
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return None, None, None, None

    fps = float(cap.get(cv2.CAP_PROP_FPS))
    frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()

    if fps <= 0:
        fps = None
    if frames <= 0:
        frames = None
    if width <= 0:
        width = None
    if height <= 0:
        height = None

    return fps, frames, width, height

<>:99: SyntaxWarning: invalid escape sequence '\,'
<>:99: SyntaxWarning: invalid escape sequence '\,'
<>:99: SyntaxWarning: invalid escape sequence '\,'
<>:99: SyntaxWarning: invalid escape sequence '\,'
C:\Users\iwo\AppData\Local\Temp\ipykernel_2820\2602998827.py:99: SyntaxWarning: invalid escape sequence '\,'
  vf = f"select='between(n\,{start_f}\,{end_f})',setpts=N/FRAME_RATE/TB"
C:\Users\iwo\AppData\Local\Temp\ipykernel_2820\2602998827.py:99: SyntaxWarning: invalid escape sequence '\,'
  vf = f"select='between(n\,{start_f}\,{end_f})',setpts=N/FRAME_RATE/TB"


In [21]:
check_required_tools()

df_train = load_split(TRAIN_JSON, 'train')
df_val = load_split(VAL_JSON, 'val')
df_test = load_split(TEST_JSON, 'test')

df = pd.concat([df_train, df_val, df_test], ignore_index=True)
df['clean_text'] = df['clean_text'].astype(str).str.strip().str.lower()
df['url_norm'] = df['url'].apply(normalize_url)

for c in ['start', 'end']:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for c in ['start_time', 'end_time', 'fps']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

base = df.head(COMPARE_ROWS).copy().reset_index(drop=True)
base['source_row_index'] = base.index
base['pair_id'] = base.apply(make_pair_id, axis=1)

frames_df = base.copy()
frames_df['trim_mode'] = 'frames'

seconds_df = base.copy()
seconds_df['trim_mode'] = 'seconds'

pilot = pd.concat([frames_df, seconds_df], ignore_index=True)
pilot['clip_id'] = pilot.apply(make_clip_id, axis=1)

OUTPUT_VIDEO_DIR.mkdir(parents=True, exist_ok=True)

print('Base rows for comparison:', len(base))
print('Total jobs (frames + seconds):', len(pilot))
print(pilot['trim_mode'].value_counts().to_dict())
display(pilot[['source_row_index', 'pair_id', 'clip_id', 'split', 'clean_text', 'url_norm', 'start', 'end', 'start_time', 'end_time', 'trim_mode']].head(20))

Tools OK: yt-dlp, ffmpeg
Base rows for comparison: 10
Total jobs (frames + seconds): 20
{'frames': 10, 'seconds': 10}


,source_row_index,pair_id,clip_id,split,clean_text,url_norm,start,end,start_time,end_time,trim_mode
0,0,165d3124dde5004c,04b86a4f90f2e5a7,train,match,https://www.youtube.com/watch?v=C37R_Ix8-qs,0,83,0.000,2.767,frames
1,1,2c1581c463998473,4a436ea6425a64bd,train,fail,https://www.youtube.com/watch?v=PIsUJl8BN_I,0,74,0.000,2.960,frames
2,2,c4457c33a0a821e4,5f6dd01b571733fd,train,laugh,https://www.youtube.com/watch?v=9FdHlMOnVjg,0,31,0.000,1.034,frames
3,3,d7544caee572ba54,ee5482a042c28fce,train,book,https://www.youtube.com/watch?v=J7tP98oDxqE,0,66,0.000,2.640,frames
4,4,bcdca3dc3f50112b,8b140950b7f3400a,train,sign language,https://www.youtube.com/watch?v=N2mG9ZKjrGA,0,75,0.000,2.502,frames
5,5,be0fc7b641d4dc62,f853e28e15c11c94,train,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,33,110,1.101,3.670,frames
6,6,40a044975f43e845,8e95d44b08038dd9,train,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,140,206,4.671,6.874,frames
7,7,3c7b932870ed1d37,dcdf465288b27e9a,train,easter,https://www.youtube.com/watch?v=SVWABYmFdhs,0,116,0.000,3.920,frames
8,8,2f38455169fac575,509dc0b4df4357d7,train,boring,https://www.youtube.com/watch?v=CYx7qm62Zwo,0,71,0.000,2.840,frames
9,9,24bba3e9ff88b8fb,de98b79c62ecb2dc,train,past,https://www.youtube.com/watch?v=cJOyCgIKyeA,0,32,0.000,1.068,frames


In [22]:
def process_row(idx: int, row: pd.Series) -> dict[str, Any]:
    split = str(row.get('split', 'unknown'))
    pair_id = str(row.get('pair_id', 'unknown'))
    clip_id = str(row.get('clip_id', 'unknown'))
    trim_mode = str(row.get('trim_mode', 'seconds'))
    url = str(row.get('url_norm', '')).strip()

    output_name = f'msasl_{split}_{pair_id}_{trim_mode}.mp4'
    output_path = OUTPUT_VIDEO_DIR / output_name

    result = {
        'row_index': idx,
        'source_row_index': row.get('source_row_index', np.nan),
        'pair_id': pair_id,
        'clip_id': clip_id,
        'split': split,
        'clean_text': row.get('clean_text', ''),
        'url': row.get('url', ''),
        'url_norm': url,
        'trim_mode': trim_mode,
        'start': row.get('start', np.nan),
        'end': row.get('end', np.nan),
        'start_time': row.get('start_time', np.nan),
        'end_time': row.get('end_time', np.nan),
        'local_video_path': str(output_path),
        'has_video': False,
        'download_status': 'pending',
        'error_message': '',
        'fps_out': np.nan,
        'frame_count_out': np.nan,
        'video_width_out': np.nan,
        'video_height_out': np.nan,
        'duration_sec_out': np.nan,
    }

    if not url:
        result['download_status'] = 'invalid_url'
        result['error_message'] = 'Empty URL'
        return result

    if output_path.exists() and not OVERWRITE:
        result['download_status'] = 'already_exists'
        result['has_video'] = True
        fps, frames, width, height = probe_video(output_path)
        result['fps_out'] = fps
        result['frame_count_out'] = frames
        result['video_width_out'] = width
        result['video_height_out'] = height
        if fps and frames:
            result['duration_sec_out'] = frames / fps
        return result

    try:
        with tempfile.TemporaryDirectory(prefix='msasl_') as tmp:
            tmp_dir = Path(tmp)
            raw_video = yt_dlp_download(url, tmp_dir)

            if trim_mode == 'frames':
                if pd.isna(row.get('start')) or pd.isna(row.get('end')):
                    raise ValueError('Missing frame range for frame-based trim')
                start_f = int(row.get('start'))
                end_f = int(row.get('end'))
                trim_by_frames(raw_video, output_path, start_f, end_f)
            else:
                if pd.isna(row.get('start_time')) or pd.isna(row.get('end_time')):
                    raise ValueError('Missing time range for second-based trim')
                start_s = float(row.get('start_time'))
                end_s = float(row.get('end_time'))
                trim_by_seconds(raw_video, output_path, start_s, end_s)

        if output_path.exists():
            result['download_status'] = 'ok'
            result['has_video'] = True
            fps, frames, width, height = probe_video(output_path)
            result['fps_out'] = fps
            result['frame_count_out'] = frames
            result['video_width_out'] = width
            result['video_height_out'] = height
            if fps and frames:
                result['duration_sec_out'] = frames / fps
        else:
            result['download_status'] = 'output_missing'
            result['error_message'] = 'Output clip not found after processing'

    except Exception as exc:
        result['download_status'] = 'failed'
        result['error_message'] = str(exc)[:1200]

    return result


records: list[dict[str, Any]] = []
total = len(pilot)

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(process_row, i, row) for i, row in pilot.iterrows()]

    done = 0
    for fut in as_completed(futures):
        records.append(fut.result())
        done += 1
        if done % 5 == 0 or done == total:
            print(f'Processed {done}/{total}')

manifest_df = pd.DataFrame(records).sort_values('row_index').reset_index(drop=True)
manifest_df.to_csv(OUTPUT_MANIFEST_CSV, index=False)

print('Saved manifest:', OUTPUT_MANIFEST_CSV)
print('Output videos dir:', OUTPUT_VIDEO_DIR)
display(manifest_df.head(10))

Processed 5/20
Processed 10/20
Processed 15/20
Processed 20/20
Saved manifest: D:\college\sem_mag_1\szum\MS-ASL\msasl_download_manifest_compare10.csv
Output videos dir: D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL


,row_index,source_row_index,pair_id,clip_id,split,clean_text,url,url_norm,trim_mode,start,end,start_time,end_time,local_video_path,has_video,download_status,error_message,fps_out,frame_count_out,video_width_out,video_height_out,duration_sec_out
0,0,0,165d3124dde5004c,04b86a4f90f2e5a7,train,match,https://www.youtube.com/watch?v=C37R_Ix8-qs,https://www.youtube.com/watch?v=C37R_Ix8-qs,frames,0,83,0.000,2.767,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,True,already_exists,,30.00000,84.0,640.0,360.0,2.800000
1,1,1,2c1581c463998473,4a436ea6425a64bd,train,fail,https://www.youtube.com/watch?v=PIsUJl8BN_I,https://www.youtube.com/watch?v=PIsUJl8BN_I,frames,0,74,0.000,2.960,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,True,already_exists,,25.00000,75.0,480.0,360.0,3.000000
2,2,2,c4457c33a0a821e4,5f6dd01b571733fd,train,laugh,www.youtube.com/watch?v=9FdHlMOnVjg,https://www.youtube.com/watch?v=9FdHlMOnVjg,frames,0,31,0.000,1.034,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,False,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN
3,3,3,d7544caee572ba54,ee5482a042c28fce,train,book,https://www.youtube.com/watch?v=J7tP98oDxqE,https://www.youtube.com/watch?v=J7tP98oDxqE,frames,0,66,0.000,2.640,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,True,already_exists,,25.00000,67.0,480.0,360.0,2.680000
4,4,4,bcdca3dc3f50112b,8b140950b7f3400a,train,sign language,www.youtube.com/watch?v=N2mG9ZKjrGA,https://www.youtube.com/watch?v=N2mG9ZKjrGA,frames,0,75,0.000,2.502,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,True,already_exists,,29.97003,76.0,640.0,360.0,2.535867
5,5,5,be0fc7b641d4dc62,f853e28e15c11c94,train,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,https://www.youtube.com/watch?v=1AyT77LqJzQ,frames,33,110,1.101,3.670,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,False,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN
6,6,6,40a044975f43e845,8e95d44b08038dd9,train,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,https://www.youtube.com/watch?v=1AyT77LqJzQ,frames,140,206,4.671,6.874,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,False,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN
7,7,7,3c7b932870ed1d37,dcdf465288b27e9a,train,easter,https://www.youtube.com/watch?v=SVWABYmFdhs,https://www.youtube.com/watch?v=SVWABYmFdhs,frames,0,116,0.000,3.920,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,True,already_exists,,29.59500,117.0,640.0,360.0,3.953371
8,8,8,2f38455169fac575,509dc0b4df4357d7,train,boring,https://www.youtube.com/watch?v=CYx7qm62Zwo,https://www.youtube.com/watch?v=CYx7qm62Zwo,frames,0,71,0.000,2.840,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,True,already_exists,,25.00000,72.0,640.0,360.0,2.880000
9,9,9,24bba3e9ff88b8fb,de98b79c62ecb2dc,train,past,https://www.youtube.com/watch?v=cJOyCgIKyeA,https://www.youtube.com/watch?v=cJOyCgIKyeA,frames,0,32,0.000,1.068,D:\college\sem_mag_1\szum\MS-ASL\videos_MS_ASL...,False,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be...",NaN,NaN,NaN,NaN,NaN


In [23]:
status_counts = manifest_df['download_status'].value_counts(dropna=False)
mode_status = pd.crosstab(manifest_df['trim_mode'], manifest_df['download_status'])

print('Status counts:')
display(status_counts.to_frame('count'))

print('Status by trim mode:')
display(mode_status)

ok_df = manifest_df[manifest_df['has_video'] == True].copy()
print('Successful clips:', len(ok_df), '/', len(manifest_df))

if len(ok_df) > 0:
    print('Output clip stats (ok only):')
    display(ok_df[['fps_out', 'frame_count_out', 'duration_sec_out', 'video_width_out', 'video_height_out']].describe().T)

# Pelny status 1:1 po pair_id (niezalezny od clip_id/hash per tryb)
pair_status = manifest_df.pivot_table(
    index='pair_id',
    columns='trim_mode',
    values='download_status',
    aggfunc='first',
)
pair_status = pair_status.rename(columns={'frames': 'frames_status', 'seconds': 'seconds_status'})
pair_status['has_pair'] = pair_status['frames_status'].notna() & pair_status['seconds_status'].notna()
pair_status['comparable_success'] = pair_status['frames_status'].isin(['ok', 'already_exists']) & pair_status['seconds_status'].isin(['ok', 'already_exists'])

print('Pair status per pair_id (has_pair/comparable_success):')
display(pair_status)

failed_rows = manifest_df[~manifest_df['download_status'].isin(['ok', 'already_exists'])].copy()
if len(failed_rows) > 0:
    print('Rows failed (cannot compare durations):')
    display(failed_rows[['pair_id', 'trim_mode', 'clean_text', 'url_norm', 'download_status', 'error_message']])

# Porownanie dlugosci tylko tam, gdzie obie metody sa OK
both_ok = manifest_df[manifest_df['download_status'].isin(['ok', 'already_exists'])].pivot_table(
    index='pair_id',
    columns='trim_mode',
    values='duration_sec_out',
    aggfunc='first',
)

if {'frames', 'seconds'}.issubset(set(both_ok.columns)):
    both_ok = both_ok.dropna(subset=['frames', 'seconds']).copy()
    both_ok['abs_duration_diff_sec'] = (both_ok['frames'] - both_ok['seconds']).abs()
    print('Rows with both methods successful (duration comparison):')
    display(both_ok.sort_values('abs_duration_diff_sec', ascending=False))

Status counts:


,count
download_status,
already_exists,12
failed,8


Status by trim mode:


download_status,already_exists,failed
trim_mode,,
frames,6,4
seconds,6,4


Successful clips: 12 / 20
Output clip stats (ok only):


...

trim_mode,frames_status,seconds_status,has_pair,comparable_success
pair_id,,,,
165d3124dde5004c,already_exists,already_exists,True,True
24bba3e9ff88b8fb,failed,failed,True,False
2c1581c463998473,already_exists,already_exists,True,True
2f38455169fac575,already_exists,already_exists,True,True
3c7b932870ed1d37,already_exists,already_exists,True,True
40a044975f43e845,failed,failed,True,False
bcdca3dc3f50112b,already_exists,already_exists,True,True
be0fc7b641d4dc62,failed,failed,True,False
c4457c33a0a821e4,failed,failed,True,False


Rows failed (cannot compare durations):


,pair_id,trim_mode,clean_text,url_norm,download_status,error_message
2,c4457c33a0a821e4,frames,laugh,https://www.youtube.com/watch?v=9FdHlMOnVjg,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
5,be0fc7b641d4dc62,frames,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
6,40a044975f43e845,frames,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
9,24bba3e9ff88b8fb,frames,past,https://www.youtube.com/watch?v=cJOyCgIKyeA,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
12,c4457c33a0a821e4,seconds,laugh,https://www.youtube.com/watch?v=9FdHlMOnVjg,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
15,be0fc7b641d4dc62,seconds,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
16,40a044975f43e845,seconds,school,https://www.youtube.com/watch?v=1AyT77LqJzQ,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."
19,24bba3e9ff88b8fb,seconds,past,https://www.youtube.com/watch?v=cJOyCgIKyeA,failed,"Command '['yt-dlp', '--no-playlist', '-f', 'be..."


Rows with both methods successful (duration comparison):


trim_mode,frames,seconds,abs_duration_diff_sec
pair_id,,,
2c1581c463998473,3.000000,2.960000,0.040000
2f38455169fac575,2.880000,2.840000,0.040000
d7544caee572ba54,2.680000,2.640000,0.040000
bcdca3dc3f50112b,2.535867,2.502500,0.033367
165d3124dde5004c,2.800000,2.800000,0.000000
3c7b932870ed1d37,3.953371,3.953371,0.000000


## Co dalej

Po pilocie porownaj skutecznosc i jakosc podejscia `frames` vs `seconds`,
a nastepnie ustaw finalna strategia dla pelnego pobrania zbioru.